# 02. Prétraitement des données (Data Preprocessing)

Ce notebook transforme les données brutes de `data/raw/` en jeux d'entraînement et de test prêts pour les modèles. Il suit la démarche du rapport de TP (chapitre 1 *Raw Data Cleaning* et chapitre 3 *Data Preprocessing*).

| Étape | Contenu |
|-------|---------|
| **0** | Chargement et contrôles qualité |
| **1** | Nettoyage : texte, valeurs bruitées, conversions, binaires, ordinales, variable dérivée, doublons, colonnes inutiles |
| **2** | Cible, colonnes à exclure (fuite d'information) |
| **3** | Valeurs manquantes, encodage one-hot |
| **4** | Découpage train / test |
| **5** | Imputation numérique et normalisation (ajustées sur le train uniquement) |
| **6** | Sauvegarde dans `data/processed/` |

L'exploration (graphiques, corrélations) est faite dans `01_data_understanding_eda.ipynb` et les modèles dans les notebooks suivants.

In [2]:
import os
import re
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

# ---------------- Configuration ----------------
DATA_FOLDER   = '../data/raw'
PROCESSED_DIR = '../data/processed'
TARGET        = 'compatible'                      # la colonne à prédire
LEAKAGE_COLS  = ['compatibility_score', 'relationship_longevity_months', 'pair_id']
AGE_MIN, AGE_MAX = 0, 120                         # bornes d'âge plausibles (rapport de TP)
TEST_SIZE     = 0.2
RANDOM_STATE  = 42


def text_columns(frame):
    """Colonnes texte / catégorielles (ni numériques, ni booléennes)."""
    return [c for c in frame.columns
            if not pd.api.types.is_numeric_dtype(frame[c]) and not pd.api.types.is_bool_dtype(frame[c])]

## Étape 0 : Chargement et contrôles qualité

In [3]:
csv_files = [f for f in os.listdir(DATA_FOLDER) if f.lower().endswith('.csv')]
print('CSV files found:', csv_files)

# On utilise le premier CSV (remplacer par le nom exact s'il y en a plusieurs)
df = pd.read_csv(os.path.join(DATA_FOLDER, csv_files[0]))
raw_shape = df.shape

print('Shape:', df.shape)
print('\nData types and non-null counts:')
df.info()
print('\nDuplicate rows:', df.duplicated().sum())
print('\nMissing values per column:')
print(df.isnull().sum().sort_values(ascending=False))
df.head()

CSV files found: ['cupid_algorithm_dataset.csv']
Shape: (100000, 30)

Data types and non-null counts:
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 30 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   pair_id                        100000 non-null  int64  
 1   a_age                          100000 non-null  int64  
 2   a_education                    100000 non-null  int64  
 3   a_location                     100000 non-null  str    
 4   a_career_field                 100000 non-null  str    
 5   a_career_ambition              100000 non-null  float64
 6   a_openness                     100000 non-null  float64
 7   a_extraversion                 100000 non-null  float64
 8   a_agreeableness                100000 non-null  float64
 9   a_conscientiousness            100000 non-null  float64
 10  a_chronotype                   100000 non-null  float64
 11  a

,pair_id,a_age,a_education,a_location,a_career_field,a_career_ambition,a_openness,a_extraversion,a_agreeableness,a_conscientiousness,a_chronotype,a_spontaneity,a_love_language,a_emotional_expressiveness,b_age,b_education,b_location,b_career_field,b_career_ambition,b_openness,b_extraversion,b_agreeableness,b_conscientiousness,b_chronotype,b_spontaneity,b_love_language,b_emotional_expressiveness,compatibility_score,compatible,relationship_longevity_months
0,1,46,3,Suburban,Healthcare,0.23,0.67,0.78,0.32,0.49,0.02,0.38,Receiving Gifts,0.59,20,3,Urban,Law,0.77,0.35,0.61,0.67,0.50,0.20,0.19,Quality Time,0.73,43.5,0,60
1,2,32,2,Suburban,Tech,0.58,0.78,0.70,0.51,0.71,0.43,0.72,Quality Time,0.73,38,1,Urban,Law,0.44,0.71,0.31,0.20,0.57,0.45,0.56,Physical Touch,0.84,60.4,0,59
2,3,25,4,Rural,Marketing,0.59,0.33,0.87,0.64,0.82,0.50,0.57,Acts of Service,0.69,46,2,Rural,Finance,0.61,0.28,0.30,0.49,0.43,0.84,0.74,Physical Touch,0.48,74.3,1,84
3,4,38,4,Suburban,Finance,0.54,0.34,0.28,0.72,0.81,0.43,0.60,Physical Touch,0.47,20,2,Urban,Science,0.39,0.47,0.35,0.46,0.21,0.80,0.35,Receiving Gifts,0.41,58.0,0,70
4,5,36,2,Rural,Entrepreneurship,0.56,0.35,0.62,0.27,0.73,0.32,0.63,Acts of Service,0.44,22,3,Urban,Creative Arts,0.63,0.44,0.66,0.45,0.43,0.86,0.36,Acts of Service,0.50,69.8,1,68


In [4]:
# --- Contrôles qualité ---
missing = df.isnull().sum()
print('Colonnes avec valeurs manquantes :\n', missing[missing > 0])
print('Total manquantes :', missing.sum())

print('\nDuplicate pair_id:', df['pair_id'].duplicated().sum() if 'pair_id' in df.columns else 'colonne absente')

print('\nPlages de valeurs des colonnes numériques (valeurs impossibles ?) :')
print(df.describe().T[['min', 'max', 'mean']])

for c in text_columns(df):
    print(f'\n{c}: {df[c].nunique()} unique ->', df[c].unique()[:10])

Colonnes avec valeurs manquantes :
 Series([], dtype: int64)
Total manquantes : 0

Duplicate pair_id: 0

Plages de valeurs des colonnes numériques (valeurs impossibles ?) :
                                min       max          mean
pair_id                         1.0  100000.0  50000.500000
a_age                          18.0      55.0     36.558140
a_education                     1.0       5.0      2.951980
a_career_ambition               0.0       1.0      0.496463
a_openness                      0.0       1.0      0.499959
a_extraversion                  0.0       1.0      0.500131
a_agreeableness                 0.0       1.0      0.501397
a_conscientiousness             0.0       1.0      0.498643
a_chronotype                    0.0       1.0      0.499697
a_spontaneity                   0.0       1.0      0.500012
a_emotional_expressiveness      0.0       1.0      0.499660
b_age                          18.0      55.0     36.515540
b_education                     1.0       5.0  

## Étape 1 : Nettoyage

Chaque anomalie est traitée dans un ordre logique. Le code détecte automatiquement les types de colonnes : si une anomalie n'existe pas dans le fichier Cupid, la cellule l'indique simplement.

### 1.1 Standardiser le texte

Une orthographe ou une casse différente (`"Male"`, `"male "`) crée de fausses catégories. On supprime les espaces superflus, on met en minuscules, puis on applique un dictionnaire de corrections optionnel.

In [5]:
text_cols = text_columns(df)
print('Colonnes texte :', text_cols)

CORRECTIONS = {
    # 'e.coi': 'e.coli',   # exemple : dictionnaire de corrections de typos
}

for col in text_cols:
    df[col] = (df[col].str.strip()
                      .str.replace(r'\s+', ' ', regex=True)
                      .str.lower())
if CORRECTIONS:
    df[text_cols] = df[text_cols].replace(CORRECTIONS)

for col in text_cols:
    n = df[col].nunique()
    if n <= 15:
        print(f'{col} ({n}) :', sorted(df[col].dropna().unique().tolist()))
    else:
        print(f'{col} : {n} valeurs uniques')

Colonnes texte : ['a_location', 'a_career_field', 'a_love_language', 'b_location', 'b_career_field', 'b_love_language']
a_location (3) : ['rural', 'suburban', 'urban']
a_career_field (10) : ['creative arts', 'education', 'engineering', 'entrepreneurship', 'finance', 'healthcare', 'law', 'marketing', 'science', 'tech']
a_love_language (5) : ['acts of service', 'physical touch', 'quality time', 'receiving gifts', 'words of affirmation']
b_location (3) : ['rural', 'suburban', 'urban']
b_career_field (10) : ['creative arts', 'education', 'engineering', 'entrepreneurship', 'finance', 'healthcare', 'law', 'marketing', 'science', 'tech']
b_love_language (5) : ['acts of service', 'physical touch', 'quality time', 'receiving gifts', 'words of affirmation']


### 1.2 Normaliser les valeurs bruitées

Les symboles et mots de remplissage (`?`, `n/a`, `unknown`...) deviennent de vraies valeurs manquantes (`NaN`) pour ne pas être comptés comme des catégories.

In [6]:
PLACEHOLDERS = ['', '?', '??', 'n/a', 'nan', 'null', 'unknown', 'undefined', '-', '--', 'missing']

before = df.isna().sum().sum()
for col in text_columns(df):
    df[col] = df[col].replace(PLACEHOLDERS, np.nan)
after = df.isna().sum().sum()
print(f'Valeurs bruitées converties en NaN : {after - before}')

Valeurs bruitées converties en NaN : 0


### 1.3 Colonnes combinées, nombres en texte, bornes d'âge

(a) Une colonne de type `x/y` est séparée en deux. (b) Les colonnes presque entièrement numériques stockées en texte sont converties. (c) Les âges hors de 0 à 120 sont mis à `NaN`.

In [7]:
combined_cols = []
for col in text_columns(df):
    s = df[col].dropna().astype(str)
    if len(s) and s.str.fullmatch(r'[^/]+/[^/]+').mean() >= 0.5:
        parts = df[col].str.split('/', expand=True)
        first = pd.to_numeric(parts[0], errors='coerce')
        df[f'{col}_1'] = first if first.notna().mean() > 0.5 else parts[0]
        df[f'{col}_2'] = parts[1]
        combined_cols.append(col)
print('Colonnes combinées séparées :', combined_cols if combined_cols else 'aucune')

converted = []
for col in text_columns(df):
    num = pd.to_numeric(df[col], errors='coerce')
    non_null = df[col].notna().sum()
    if non_null and num.notna().sum() / non_null >= 0.95:
        df[col] = num
        converted.append(col)
print('Colonnes texte converties en numérique :', converted if converted else 'aucune')

age_cols = [c for c in df.columns
            if re.search(r'(^|_)age($|_)', c.lower()) and pd.api.types.is_numeric_dtype(df[c])]
for col in age_cols:
    bad = (df[col] < AGE_MIN) | (df[col] > AGE_MAX)
    df.loc[bad, col] = np.nan
    print(f'{col} : {bad.sum()} valeurs hors bornes mises à NaN | plage {df[col].min()} - {df[col].max()}')

Colonnes combinées séparées : aucune
Colonnes texte converties en numérique : aucune
a_age : 0 valeurs hors bornes mises à NaN | plage 18.0 - 55.0
b_age : 0 valeurs hors bornes mises à NaN | plage 18.0 - 55.0


### 1.4 Recoder les variables binaires (Yes / No)

Les représentations synonymes (`1/0`, `True/False`, `Y/N`) sont unifiées en `Yes` / `No`. Les colonnes numériques 0/1 ne sont pas touchées.

In [8]:
YES = {'yes', 'y', 'true', 't', '1', '1.0'}
NO  = {'no', 'n', 'false', 'f', '0', '0.0'}

def to_yes_no(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    return 'Yes' if x in YES else ('No' if x in NO else np.nan)

binary_recoded = []
for col in df.columns:
    if col == TARGET:
        continue
    if pd.api.types.is_bool_dtype(df[col]):
        df[col] = df[col].map({True: 'Yes', False: 'No'})
        binary_recoded.append(col)
    elif col in text_columns(df):
        vals = {str(v).strip().lower() for v in df[col].dropna().unique()}
        if vals and vals <= (YES | NO):
            df[col] = df[col].apply(to_yes_no)
            binary_recoded.append(col)
print('Colonnes binaires recodées :', binary_recoded if binary_recoded else 'aucune')

Colonnes binaires recodées : aucune


### 1.5 Encoder les variables ordinales

Une variable ordinale (ex. *never < rarely < often*) est représentée par des entiers pour conserver l'ordre. Les échelles connues sont détectées automatiquement ; on peut en ajouter dans `CUSTOM_ORDINAL`.

In [9]:
KNOWN_SCALES = [
    {'never': 0, 'rarely': 1, 'regularly': 2, 'often': 3},
    {'never': 0, 'rarely': 1, 'sometimes': 2, 'often': 3, 'very often': 4, 'always': 5},
    {'very low': 0, 'low': 1, 'medium': 2, 'high': 3, 'very high': 4},
    {'strongly disagree': 0, 'disagree': 1, 'neutral': 2, 'agree': 3, 'strongly agree': 4},
]
CUSTOM_ORDINAL = {
    # 'une_colonne': {'low': 0, 'medium': 1, 'high': 2},
}

ordinal_encoded = {}
for col, mapping in CUSTOM_ORDINAL.items():
    if col in df.columns:
        df[col] = df[col].map(mapping)
        ordinal_encoded[col] = mapping

for col in text_columns(df):
    vals = set(df[col].dropna().unique())
    for scale in KNOWN_SCALES:
        if len(vals) >= 2 and vals <= set(scale):
            df[col] = df[col].map(scale)
            ordinal_encoded[col] = scale
            break

print('Colonnes ordinales encodées :', list(ordinal_encoded) if ordinal_encoded else 'aucune')

Colonnes ordinales encodées : aucune


### 1.6 Variable dérivée

Équivalent de l'indicateur *MultiResistance* du TP : l'écart d'âge entre les deux partenaires, `age_gap = |a_age - b_age|`.

In [10]:
if {'a_age', 'b_age'} <= set(df.columns):
    df['age_gap'] = (df['a_age'] - df['b_age']).abs()
    print('age_gap créée. Écart moyen : {:.2f} ans'.format(df['age_gap'].mean()))
else:
    print("Colonnes 'a_age' / 'b_age' absentes : aucune variable dérivée créée.")

age_gap créée. Écart moyen : 12.64 ans


### 1.7 Supprimer les doublons

Les doublons gonflent artificiellement certains cas et biaisent les modèles. Cette étape est faite **avant** la suppression des identifiants, pour ne pas fusionner par erreur deux paires différentes qui auraient les mêmes caractéristiques.

In [11]:
initial = len(df)
n_dup = df.duplicated().sum()
print('Lignes dupliquées :', n_dup)
df = df.drop_duplicates().reset_index(drop=True)
print(f'{initial - len(df)} lignes supprimées.')

if 'pair_id' in df.columns:
    print('pair_id dupliqués restants :', df['pair_id'].duplicated().sum())

Lignes dupliquées : 0
0 lignes supprimées.
pair_id dupliqués restants : 0


### 1.8 Supprimer les colonnes inutiles

Identifiants, noms, e-mails, adresses, notes et dates n'ont aucune valeur prédictive. Les colonnes constantes et les colonnes combinées déjà séparées sont aussi retirées. La cible n'est jamais supprimée.

In [12]:
META_WORDS = {'id', 'name', 'email', 'address', 'phone', 'note', 'notes', 'comment', 'comments', 'timestamp', 'date'}

irrelevant = [c for c in df.columns
              if c != TARGET and (set(re.split(r'[_\s/]+', c.lower())) & META_WORDS)]
constant   = [c for c in df.columns if c != TARGET and df[c].nunique(dropna=True) <= 1]
irrelevant = list(dict.fromkeys(irrelevant + constant + combined_cols))

print('Colonnes supprimées :', irrelevant if irrelevant else 'aucune')
df = df.drop(columns=[c for c in irrelevant if c in df.columns])
print('Shape brute :', raw_shape, '-> nettoyée :', df.shape)

Colonnes supprimées : ['pair_id']
Shape brute : (100000, 30) -> nettoyée : (100000, 30)


In [13]:
# Sauvegarde du jeu nettoyé (nom distinct des fichiers produits par src/)
os.makedirs(PROCESSED_DIR, exist_ok=True)
df.to_csv(os.path.join(PROCESSED_DIR, 'cupid_cleaned.csv'), index=False)
print('Jeu nettoyé sauvegardé : cupid_cleaned.csv')

Jeu nettoyé sauvegardé : cupid_cleaned.csv


## Étape 2 : Cible et colonnes à exclure

`compatibility_score`, `relationship_longevity_months` et `pair_id` ne doivent pas servir de variables explicatives : les deux premières sont liées directement au résultat (fuite d'information) et la dernière est un identifiant.

In [14]:
print(df[TARGET].value_counts())
print(df[TARGET].value_counts(normalize=True).round(3))

X = df.drop(columns=[c for c in LEAKAGE_COLS if c in df.columns] + [TARGET])   # les indices
y = df[TARGET]                                                                  # la réponse à deviner

for c in LEAKAGE_COLS:
    assert c not in X.columns, f"'{c}' ne doit pas rester dans X"
print('Features shape:', X.shape)

compatible
0    57119
1    42881
Name: count, dtype: int64
compatible
0    0.571
1    0.429
Name: proportion, dtype: float64
Features shape: (100000, 27)


## Étape 3 : Valeurs manquantes catégorielles et encodage one-hot

Les valeurs manquantes des colonnes texte deviennent une catégorie `missing` (aucun calcul appris sur les données, donc pas de fuite). Ensuite les colonnes texte sont transformées en indicatrices 0/1. Les valeurs manquantes numériques sont traitées après le découpage (étape 5).

In [15]:
cat_cols = text_columns(X)
print('Colonnes catégorielles :', cat_cols)

X[cat_cols] = X[cat_cols].fillna('missing')
for c in cat_cols:
    print(f'{c}: {X[c].nunique()} valeurs ->', X[c].unique()[:8])

X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True, dtype=int)
print('\nAvant :', X.shape, '-> Après :', X_encoded.shape)
X_encoded.head()

Colonnes catégorielles : ['a_location', 'a_career_field', 'a_love_language', 'b_location', 'b_career_field', 'b_love_language']
a_location: 3 valeurs -> <StringArray>
['suburban', 'rural', 'urban']
Length: 3, dtype: str
a_career_field: 10 valeurs -> <StringArray>
['healthcare', 'tech', 'marketing', 'finance', 'entrepreneurship', 'science', 'engineering', 'law']
Length: 8, dtype: str
a_love_language: 5 valeurs -> <StringArray>
['receiving gifts', 'quality time', 'acts of service', 'physical touch', 'words of affirmation']
Length: 5, dtype: str
b_location: 3 valeurs -> <StringArray>
['urban', 'rural', 'suburban']
Length: 3, dtype: str
b_career_field: 10 valeurs -> <StringArray>
['law', 'finance', 'science', 'creative arts', 'education', 'marketing', 'healthcare', 'tech']
Length: 8, dtype: str
b_love_language: 5 valeurs -> <StringArray>
['quality time', 'physical touch', 'receiving gifts', 'acts of service', 'words of affirmation']
Length: 5, dtype: str

Avant : (100000, 27) -> Après : (1

,a_age,a_education,a_career_ambition,a_openness,a_extraversion,a_agreeableness,a_conscientiousness,a_chronotype,a_spontaneity,a_emotional_expressiveness,b_age,b_education,b_career_ambition,b_openness,b_extraversion,b_agreeableness,b_conscientiousness,b_chronotype,b_spontaneity,b_emotional_expressiveness,age_gap,a_location_suburban,a_location_urban,a_career_field_education,a_career_field_engineering,a_career_field_entrepreneurship,a_career_field_finance,a_career_field_healthcare,a_career_field_law,a_career_field_marketing,a_career_field_science,a_career_field_tech,a_love_language_physical touch,a_love_language_quality time,a_love_language_receiving gifts,a_love_language_words of affirmation,b_location_suburban,b_location_urban,b_career_field_education,b_career_field_engineering,b_career_field_entrepreneurship,b_career_field_finance,b_career_field_healthcare,b_career_field_law,b_career_field_marketing,b_career_field_science,b_career_field_tech,b_love_language_physical touch,b_love_language_quality time,b_love_language_receiving gifts,b_love_language_words of affirmation
0,46.0,3,0.23,0.67,0.78,0.32,0.49,0.02,0.38,0.59,20.0,3,0.77,0.35,0.61,0.67,0.50,0.20,0.19,0.73,26.0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0
1,32.0,2,0.58,0.78,0.70,0.51,0.71,0.43,0.72,0.73,38.0,1,0.44,0.71,0.31,0.20,0.57,0.45,0.56,0.84,6.0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0
2,25.0,4,0.59,0.33,0.87,0.64,0.82,0.50,0.57,0.69,46.0,2,0.61,0.28,0.30,0.49,0.43,0.84,0.74,0.48,21.0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0
3,38.0,4,0.54,0.34,0.28,0.72,0.81,0.43,0.60,0.47,20.0,2,0.39,0.47,0.35,0.46,0.21,0.80,0.35,0.41,18.0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0
4,36.0,2,0.56,0.35,0.62,0.27,0.73,0.32,0.63,0.44,22.0,3,0.63,0.44,0.66,0.45,0.43,0.86,0.36,0.50,14.0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0


## Étape 4 : Découpage train / test

Découpage stratifié 80 % / 20 % pour conserver la même proportion de compatibles dans les deux jeux.

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print('Train :', X_train.shape, '| Test :', X_test.shape)
print('\nProportion dans train :\n', y_train.value_counts(normalize=True).round(3))
print('\nProportion dans test :\n', y_test.value_counts(normalize=True).round(3))

Train : (80000, 51) | Test : (20000, 51)

Proportion dans train :
 compatible
0    0.571
1    0.429
Name: proportion, dtype: float64

Proportion dans test :
 compatible
0    0.571
1    0.429
Name: proportion, dtype: float64


## Étape 5 : Imputation numérique et normalisation

La médiane (imputation) et la moyenne / l'écart-type (normalisation) sont **calculés sur le train uniquement**, puis appliqués au test, afin d'éviter toute fuite d'information.

In [17]:
# Colonnes numériques d'origine (sans les colonnes 0/1 issues de l'encodage)
num_cols = X.select_dtypes(include='number').columns.tolist()
print('Colonnes à imputer / normaliser :', len(num_cols))
print('Valeurs manquantes numériques (train) :', int(X_train[num_cols].isna().sum().sum()))

imputer = SimpleImputer(strategy='median')
scaler  = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()

X_train_scaled[num_cols] = imputer.fit_transform(X_train[num_cols])
X_test_scaled[num_cols]  = imputer.transform(X_test[num_cols])

X_train_scaled[num_cols] = scaler.fit_transform(X_train_scaled[num_cols])
X_test_scaled[num_cols]  = scaler.transform(X_test_scaled[num_cols])

assert not X_train_scaled.isna().any().any() and not X_test_scaled.isna().any().any(), 'Valeurs manquantes restantes'
X_train_scaled[num_cols].describe().T[['mean', 'std']].head()

Colonnes à imputer / normaliser : 21
Valeurs manquantes numériques (train) : 0


,mean,std
a_age,1.630696e-16,1.000006
a_education,2.309264e-18,1.000006
a_career_ambition,-5.575984e-16,1.000006
a_openness,-1.413980e-16,1.000006
a_extraversion,-3.527845e-16,1.000006


## Étape 6 : Sauvegarde

Les quatre fichiers ci-dessous sont ceux lus par la suite du projet.

In [18]:
os.makedirs(PROCESSED_DIR, exist_ok=True)

X_train_scaled.to_csv(os.path.join(PROCESSED_DIR, 'X_train.csv'), index=False)
X_test_scaled.to_csv(os.path.join(PROCESSED_DIR, 'X_test.csv'), index=False)
y_train.to_csv(os.path.join(PROCESSED_DIR, 'y_train.csv'), index=False)
y_test.to_csv(os.path.join(PROCESSED_DIR, 'y_test.csv'), index=False)

print('Fichiers sauvegardés dans data/processed :', os.listdir(PROCESSED_DIR))
print('X_train :', X_train_scaled.shape, '| X_test :', X_test_scaled.shape)

Fichiers sauvegardés dans data/processed : ['cupid_algorithm_dataset.csv', 'cupid_cleaned.csv', 'cupid_cleaned_eda.csv', 'X_test.csv', 'X_train.csv', 'y_test.csv', 'y_train.csv']
X_train : (80000, 51) | X_test : (20000, 51)
